<a href="https://colab.research.google.com/github/Nyfeu/Data-Science/blob/main/Ex_reshape.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<head>
  <meta name="author" content="Rogério de Oliveira">
  <meta institution="author" content="ITM">
</head>

<img src="https://maua.br/images/selo-60-anos-maua.svg" width=300, align="right">
<!-- <h1 align=left><font size = 6, style="color:rgb(200,0,0)"> optional title </font></h1> -->


In [43]:
import numpy as np
import pandas as pd

# **CASE: Yahoo Finance**

Nos exercícios a seguir partimos da seguinte base de dados extraída do `yahoo finance`:

In [44]:
df = pd.read_csv('https://github.com/Rogerio-mack/IMT_CD_2026/raw/refs/heads/main/data/stocks_2019_2025.csv')
df = df[['Date', 'BRL=X', 'ITUB3.SA', 'PETR4.SA', 'VALE3.SA', '^BVSP', 'BTC-USD']]
df.head()

,Date,BRL=X,ITUB3.SA,PETR4.SA,VALE3.SA,^BVSP,BTC-USD
0,2019-01-01,3.8800,NaN,NaN,NaN,NaN,3843.520020
1,2019-01-02,3.8799,20.093735,7.466959,30.699385,91012.0,3943.409424
2,2019-01-03,3.7863,20.276070,7.650064,29.443529,91564.0,3836.741211
3,2019-01-04,3.7551,19.936563,7.671789,31.360359,91841.0,3857.717529
4,2019-01-05,NaN,NaN,NaN,NaN,NaN,3845.194580


# Q1.

Empregue os métodos de reshape do `pandas` e a função `pd.to_datetime(df.Date).dt.day_name()` para transformar o dataframe `df` para o formato abaixo.

In [45]:
# Fazendo o 'melt' para transformar as colunas em entradas/linhas
df_melt = df.melt(id_vars=['Date'], var_name='Ticker', value_name='Adj Close')

# Adicionando a coluna 'day_of_week'
df_melt['day_of_week'] = pd.to_datetime(df_melt.Date).dt.day_name()

display(df_melt)

,Date,Ticker,Adj Close,day_of_week
0,2019-01-01,BRL=X,3.880000,Tuesday
1,2019-01-02,BRL=X,3.879900,Wednesday
2,2019-01-03,BRL=X,3.786300,Thursday
3,2019-01-04,BRL=X,3.755100,Friday
4,2019-01-05,BRL=X,NaN,Saturday
...,...,...,...,...
13609,2025-03-14,BTC-USD,83969.101562,Friday
13610,2025-03-15,BTC-USD,84343.109375,Saturday
13611,2025-03-16,BTC-USD,82579.687500,Sunday
13612,2025-03-17,BTC-USD,84075.687500,Monday


In [46]:
#@markdown Must be True
len(df_melt[ df_melt.day_of_week == 'Tuesday' ]) == 1950

True

Qual o maior valor do Real às sextas-feiras (`Friday`) dentro do período e em que data isso ocorreu? (Atenção, a base contém cotações do dólar).

In [47]:
result = df_melt[(df_melt['day_of_week'] == 'Friday') & (df_melt['Ticker'] == 'BRL=X')].min()
result

,0
Date,2019-01-04
Ticker,BRL=X
Adj Close,3.6428
day_of_week,Friday


In [48]:
result = df_melt.loc[df_melt['day_of_week'] == 'Friday']
result = result.loc[result['Ticker'] == 'BRL=X']
result = result.loc[result['Adj Close'] == result['Adj Close'].min()]
result

,Date,Ticker,Adj Close,day_of_week
31,2019-02-01,BRL=X,3.6428,Friday


## Q2.

Empregue os métodos de reshape do `pandas` remodelar o dataframe criado no exercício anterior produzindo o dataframe `df_Friday` que contém somente valores de fechamento da semana (`Friday`) dos ativos como no formato abaixo.

In [49]:
df_Friday = df_melt[df_melt['day_of_week'] == 'Friday'].pivot(index='Date', columns='Ticker', values='Adj Close')
display(df_Friday)

Ticker,BRL=X,BTC-USD,ITUB3.SA,PETR4.SA,VALE3.SA,^BVSP
Date,,,,,,
2019-01-04,3.7551,3857.717529,19.936563,7.671789,31.360359,91841.0
2019-01-11,3.7079,3687.365479,20.200619,7.755583,31.474533,93658.0
2019-01-18,3.7489,3657.839355,20.307501,7.882824,32.904644,96097.0
2019-01-25,3.7698,3599.765869,NaN,NaN,NaN,NaN
2019-02-01,3.6428,3487.945312,20.568373,8.006965,27.791084,97861.0
...,...,...,...,...,...,...
2025-02-14,5.7669,97508.968750,26.610332,37.439999,53.578861,128219.0
2025-02-21,5.7013,96125.546875,26.438196,38.389999,55.975330,127128.0
2025-02-28,5.8390,84373.007812,25.947592,35.930000,53.078396,122799.0


In [50]:
#@markdown Must be True
int(df_Friday['BRL=X'].sum()) == 1630

True

Qual a média da cotação do dólar nas últimas 4 sextas-feiras da base?

In [51]:
df_friday_dolar = df_Friday['BRL=X'].sort_values(ascending=False).head(4)

display(df_friday_dolar)

,BRL=X
Date,
2024-12-20,6.1516
2025-01-03,6.1510
2024-12-27,6.1485
2025-01-17,6.0499


In [52]:
df_friday_dolar.mean()

np.float64(6.125249981880188)

# Q3.

Empregue df_melt para obter a data de maior valor de cada um dos ativos.

In [53]:
# Encontrando o índice do valor máximo para cada Ticket
idx_max = df_melt.groupby('Ticker')['Adj Close'].idxmax()

# Filtrando o dataframe original usando esses índices
result_max = df_melt.loc[idx_max]

display(result_max)

,Date,Ticker,Adj Close,day_of_week
2193,2025-01-02,BRL=X,6.300000,Thursday
13557,2025-01-21,BTC-USD,106146.265625,Tuesday
4537,2025-03-18,ITUB3.SA,28.450001,Tuesday
6780,2025-02-20,PETR4.SA,38.500000,Thursday
8293,2023-01-26,VALE3.SA,81.193130,Thursday
11142,2024-08-28,^BVSP,137344.000000,Wednesday
